# Caso 1: Hola Spark Streaming

Aplicación de conteo de palabras en tiempo real mediante Apache Spark Structured Streaming conectada a un socket TCP local en el puerto 9999.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

spark = SparkSession \
    .builder \
    .appName("Streaming IABD WordCount") \
    .master("local[2]") \
    .getOrCreate()

In [2]:
# Creamos un flujo de escucha sobre netcat en localhost:9999
# En Spark Streaming, la lectura se realiza mediante readStream
lineasDF = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", "9999") \
    .load()

# Leemos las líneas y las pasamos a palabras.
# Sobre ellas, realizamos la agrupación count
palabrasDF = lineasDF.select(explode(split(lineasDF.value, ' ')).alias('palabra'))
cantidadDF = palabrasDF.groupBy("palabra").count()

In [3]:
# Mostramos las palabras por consola
# En Spark Streaming, la persistencia se realiza mediante writeStream
# y en vez de hacer un save, ahora utilizamos start
wordCountQuery = cantidadDF.writeStream \
    .format("console") \
    .outputMode("complete") \
    .start()

# Dejamos Spark a la escucha
wordCountQuery.awaitTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
+-------+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
|   Esta|    1|
|     es|    1|
|     la|    1|
|primera|    1|
|  frase|    1|
+-------+-----+

-------------------------------------------
Batch: 2
-------------------------------------------
+--------+-----+
| palabra|count|
+--------+-----+
|    Esta|    1|
|      es|    1|
|      la|    1|
| primera|    2|
|   frase|    1|
|Enviamos|    1|
| segunda|    1|
|   linea|    1|
+--------+-----+
